# 4. LLM as a Judge (Answer Quality)

For answer quality, exact string match is far too strict — the RAG answer need
not copy the FAQ word-for-word, it must convey the same key info.

So we use **another LLM call as the judge**: given Q, A (ground truth) and A'
(RAG answer), output `good` / `bad` **with reasoning**.

This evaluates the *entire* RAG flow in one verdict: search + prompt + LLM. The
`reasoning` field tells us *where* it broke when it's `bad`.

In [1]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(description="Reasoning about the quality of the answer.")
    score: Literal["good", "bad"] = Field(description="'good' if correct and complete, 'bad' otherwise.")

In [2]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [3]:
from dotenv import load_dotenv
import anthropic
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress
load_dotenv()
client = anthropic.Anthropic()

In [4]:
# Load the RAG answers to judge (the glm-5.2 answers we produced in notebook 3).
import pandas as pd, json
df_answers = pd.read_csv("data/rag-answers.csv")
answers = df_answers.to_dict(orient="records")
answers[0]

{'question': 'Am I too late to enroll if I found this class just now?',
 'answer_llm': 'Yes, you can still join! According to the provided context, if you just discovered the course, you can still enroll and participate. \n\nHowever, if your goal is to earn a certificate, you need to submit your final project while the course is still accepting submissions.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [5]:
# Judge ONE record.
rec = answers[0]
prompt = aqa_judge_prompt.format(question=rec["question"], answer_orig=rec["answer_orig"], answer_llm=rec["answer_llm"])
eval_result, usage = llm_structured_retry(client, aqa_judge_instructions, prompt, AnswerEvaluation)
print(eval_result.score, "|", eval_result.reasoning)
calc_price(usage)

good | The AI answer correctly conveys the two key points from the original answer: (1) Yes, you can still enroll, and (2) to receive a certificate, you need to submit your project while submissions are still being accepted. The AI answer adds a bit of extra wording but is semantically equivalent.


{'input_cost': 0.00034125,
 'output_cost': 0.00035549999999999997,
 'total_cost': 0.0006967499999999999}

In [6]:
def evaluate_aqa(question, answer_orig, answer_llm):
    prompt = aqa_judge_prompt.format(question=question, answer_orig=answer_orig, answer_llm=answer_llm)
    return llm_structured_retry(client, aqa_judge_instructions, prompt, AnswerEvaluation)

def judge_record(rec):
    result, usage = evaluate_aqa(rec["question"], rec["answer_orig"], rec["answer_llm"])
    return ({
        "question": rec["question"], "document": rec["document"],
        "score": result.score, "reasoning": result.reasoning,
    }, usage)

> **Cost control:** set `N_JUDGE`. The course judged all 395 (~$0.25). Here we
> judge all the glm-5.2 answers produced in notebook 3.

In [7]:
from concurrent.futures import ThreadPoolExecutor

N_JUDGE = len(answers)  # judge all answers from notebook 3
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers[:N_JUDGE], judge_record)

evaluations, usages = [], []
for ev, u in results:
    evaluations.append(ev); usages.append(u)

print("judge cost: $%.6f" % calc_total_price(usages))

  0%|          | 0/50 [00:00<?, ?it/s]

judge cost: $0.044799


In [8]:
df_eval = pd.DataFrame(evaluations)
print(df_eval["score"].value_counts())
print()
print("good ratio: %.2f%%" % (100 * (df_eval["score"] == "good").mean()))
df_eval.to_csv("data/rag-evaluations.csv", index=False)
df_eval.head()

score
good    34
bad     16
Name: count, dtype: int64

good ratio: 68.00%


,question,document,score,reasoning
0,Am I too late to enroll if I found this class ...,74eb249bbf,good,The AI answer correctly conveys both key point...
1,Started late - is it still possible to sign up?,74eb249bbf,good,The AI answer conveys the exact same informati...
2,Can I hop in even though the course already st...,74eb249bbf,bad,"The AI answer says ""I don't know,"" which compl..."
3,If I register now will I still be able to earn...,74eb249bbf,good,The AI answer correctly conveys the same core ...
4,"Missed the beginning, can I still get in and g...",74eb249bbf,good,Both answers confirm that the student can stil...


In [9]:
# The 'bad' rows are the most useful part of evaluation -> inspect them.
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
2,Can I hop in even though the course already st...,74eb249bbf,bad,"The AI answer says ""I don't know,"" which compl..."
12,How do I ask questions during the live stream?,489dd1c9d9,bad,The AI answer correctly identifies the key poi...
13,Where does the livestream link get posted befo...,489dd1c9d9,bad,The AI answer correctly addresses the student'...
15,Is there a specific order I need to follow for...,04919992b3,bad,The AI answer completely fails to answer the s...
16,What's the usual routine for getting through e...,04919992b3,bad,The AI answer correctly covers the typical mod...


## Judging the judge

The judge itself can be wrong (e.g. too lenient). You can't audit it with *another*
judge — you must **read a sample of verdicts yourself** and disagree/agree, then
tighten the instructions and re-run. A Streamlit app showing Q / A / A' / verdict
side by side is the course's suggested workflow.

**Reference result (course, full 395 run):** 379 good / 16 bad (~96% good), ~$0.25.
For agents (lesson 14) the same idea applies, plus you save the **tool-call
trajectory** and judge answer quality + trajectory quality separately.